# Automated Sales Integrity & Reporting Engine
**Module:** Python Foundations for Business Data Cleaning  
**Deliverable:** Mini Project Jupyter Notebook (`.ipynb`)

---

In [1]:
import pandas as pd
import numpy as np

def run_sales_integrity_engine(file_path):
    print("=" * 65)
    print("   AUTOMATED SALES INTEGRITY & REPORTING ENGINE   ")
    print("=" * 65)
    
    # 1. Load Data
    df = pd.read_csv(file_path)
    print(f"[SUCCESS] Dataset loaded successfully. Initial Records: {len(df)}")

    # 2. String Normalization
    print("\n--- 1. NORMALIZING STRING FIELDS ---")
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        df[col] = df[col].astype(str).str.strip().str.title()
    print(f"[COMPLETED] Cleaned whitespace and applied title casing to string columns.")

    # 3. Numeric Conversion & Type Mismatch Validation
    print("\n--- 2. DATA TYPE VALIDATION & CONVERSION ---")
    numeric_cols = ['SALES', 'QUANTITYORDERED', 'PRICEEACH']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    print("[COMPLETED] Numeric columns validated and type-cast to float.")

    # 4. Identification of Revenue Outliers (IQR Method)
    print("\n--- 3. OUTLIER DETECTION (IQR METHOD) ---")
    q1 = df['SALES'].quantile(0.25)
    q3 = df['SALES'].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    outliers = df[df['SALES'] > upper_bound]
    print(f"[ANALYSIS] Q1: ${q1:,.2f} | Q3: ${q3:,.2f} | IQR: ${iqr:,.2f}")
    print(f"[ANALYSIS] Outlier Threshold (> ${upper_bound:,.2f}): {len(outliers)} records flagged.")

    # 5. Automated Console Summary Report
    print("\n--- 4. AUTOMATED CONSOLE SUMMARY REPORT ---")
    summary = df.groupby('PRODUCTLINE')['SALES'].agg(['count', 'sum', 'mean']).reset_index()
    summary.columns = ['Product Line', 'Order Count', 'Total Sales ($)', 'Average Order ($)']
    summary['Category'] = np.where(summary['Total Sales ($)'] > 100000, 'High Revenue Product', 'Standard Revenue')
    print(summary.to_string(index=False))
    print("=" * 65)

run_sales_integrity_engine('sales_data_sample.csv')